<a href="https://colab.research.google.com/github/charre2021/reasoning_agent_example/blob/main/reasoning_agent_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import List, Dict, Any
import random
from openai import OpenAI
from google.colab import userdata

class LLM:
    def __init__(self, model : str, api_key : str) -> None:
        self.client = OpenAI(api_key = api_key)
        self.model = model

    def generate(self, prompt : str) -> str:
        response = self.client.responses.create(
            model = self.model,
            input = prompt
        )
        return response.output_text

class HierarchicalGoal:
    def __init__(self, parent_goal: str) -> None:
        self.parent_goal = parent_goal
        self.sub_goals : List['HierarchicalGoal'] = []
        self.completed = False

    def need_sub_goals(self, additional_context : str, llm : LLM) -> bool:
        while True:
            additional_context = f"Goal: {self.parent_goal}\n"
            additional_context += "Recent observations:\n"
            additional_context += """\n
            Think about the current situation and the goal. Does this goal have subgoals?
            If yes, say, 'True.' If no, say, 'False.' Limit your response to one word.
            """
            try:
                print("figuring out whether subgoals are needed")
                response = bool(llm.generate(additional_context))
                break;
            except:
                print("something went wrong with figuring out whether subgoals are needed, trying again")
                additional_context += """The format of your response was incorrect. Let's try that again.
                Remember to limit your response to just 'True' or 'False'."""
                continue
        return response

    def generate_sub_goals(self, additional_context : str, llm : LLM) -> List[str]:
        SUBGOAL_DELIMITER = "NEXT SUBGOAL:"
        additional_context += f"You said that subgoals were needed for {self.parent_goal}."
        additional_context += f"Ok, can you please list all subgoals? Delimit each with '{SUBGOAL_DELIMITER}'."
        while True:
            try:
                print(f"generating subgoals with {additional_context}")
                response = list(filter(None, llm.generate(additional_context).split(f"{SUBGOAL_DELIMITER}")))
                print(response)
                break;
            except:
                print("failed to generate subgoals, trying again")
                additional_context += f"""The format of your response was incorrect. Let's try that again.
                Please list all subgoals you believe are needed for {self.parent_goal}
                Remember to list all subgoals with the '{SUBGOAL_DELIMITER}'."""
                continue
        return response

    def add_sub_goal(self, sub_goal : 'HierarchicalGoal') -> None:
        self.sub_goals.append(sub_goal)

    def mark_completed(self) -> None:
        self.completed = True

    def __str__(self) -> str:
        full_string = "Here is the parent: " + self.parent_goal + "\n"
        full_string += "Here are its subgoals:\n"
        full_string = full_string.join(x.__str__() for x in self.sub_goals)
        return full_string

class LLMAgent:
    def __init__(self, llm, action_space: List[str]) -> None:
        self.llm = llm
        self.action_space = action_space
        self.memory = []
        self.goal_stack = []
        self.current_goal = None

    def remember(self) -> str:
        return "" if len(self.memory) == 0 else "".join(self.memory[-5:])

    def add_goal(self, goal: 'HierarchicalGoal') -> None:
        self.goal_stack.append(goal)
        memory = self.remember()
        print("checking if sub goals are needed")
        are_sub_goals_needed = goal.need_sub_goals(memory, self.llm)
        if are_sub_goals_needed:
            print("sub goals are needed")
            sub_goals_list = goal.generate_sub_goals(memory, self.llm)
            for sub_goal in sub_goals_list:
                print("adding sub goals")
                print(sub_goal)
                new_sub_goal = HierarchicalGoal(sub_goal)
                goal.add_sub_goal(new_sub_goal)
                self.goal_stack.append(new_sub_goal)

    def is_goal_completed(self, goal : 'HierarchicalGoal') -> bool:
        if len(goal.sub_goals) > 0 and all([self.is_goal_completed(i) for i in goal.sub_goals]):
            print("evaluating completion of subgoals")
            goal.mark_completed()
        elif len(goal.sub_goals) == 0:
            context = self.remember()
            while True:
                try:
                    context += f"""Has {goal} been completed yet? Please respond with 'True' or 'False' only."""
                    print(f"checking completion of goal with this context: {context}")
                    response = bool(llm.generate(context))
                except:
                    print("something went wrong with figuring out whether goal is completed, trying again")
                    context += """The format of your response was incorrect. Let's try that again.
                    Remember to limit your response to just 'True' or 'False'."""
            if response:
                print(f"marking {goal} completed")
                goal.mark_completed()
        return goal.completed

    def perceive(self, observation: str) -> None:
        self.memory.append(observation)

    def think(self) -> str:
        context = f"Goal: {self.current_goal}\n"
        context += "Recent observations:\n"
        context += "\n".join(self.memory[-5:])
        context += "\nThink about the current situation and the goal. What should be done next?"

        return self.llm.generate(context)

    def decide(self, thought: str) -> str:
        context = f"Thought: {thought}\n"
        context += "Based on this thought, choose the most appropriate action from the following:\n"
        context += ", ".join(self.action_space)
        context += "\nChosen action:"

        return self.llm.generate(context)


    def act(self, action: str) -> str:
        outcomes = [
            f"Action '{action}' was successful.",
            f"Action '{action}' failed.",
            f"Action '{action}' had an unexpected outcome."
        ]
        return random.choice(outcomes)

    def run_step(self):
        thought = self.think()
        action = self.decide(thought)
        outcome = self.act(action)
        self.perceive(outcome)
        return thought, action, outcome



In [2]:
llm = LLM("o4-mini", userdata.get("openai"))
action_space = ["move", "grab", "drop", "use", "talk"]
agent = LLMAgent(llm, action_space)

g = "Find the key and unlock the door."
agent.add_goal(HierarchicalGoal(g))
print(agent.goal_stack[0])
# agent.perceive("You are in a room with a table and a chair. There's a drawer in the table.")

# for _ in range(5):
#     thought, action, outcome = agent.run_step()
#     print(f"Thought: {thought}")
#     print(f"Action: {action}")
#     print(f"Outcome: {outcome}")

checking if sub goals are needed
figuring out whether subgoals are needed
sub goals are needed
generating subgoals with You said that subgoals were needed for Find the key and unlock the door..Ok, can you please list all subgoals? Delimit each with 'NEXT SUBGOAL:'.
[' Identify the most likely locations where the key might be (e.g., tabletop, drawer, hook).  \n', ' Navigate to the first candidate location.  \n', ' Use perception sensors to scan that location for the key.  \n', ' If the key is not found, move to the next candidate location and repeat scanning.  \n', ' Once the key is detected, plan an approach trajectory to the key’s position.  \n', ' Reach out and grasp the key securely.  \n', ' Verify a stable grip on the key.  \n', ' Plan a collision-free path from the key’s location to the door.  \n', ' Navigate along that path, avoiding obstacles.  \n', ' Position yourself in front of the door lock, aligning the key with the keyhole.  \n', ' Insert the key into the keyhole.  \n', ' 